# ⚾ MLB Lineup Generator & Machine Learning Optimizer
### Comprehensive End-to-End Lineup Optimization, Simulation, and Statistical Validation

This notebook provides a complete environment for:
1. **MLB Data Ingestion**: Fetching real-time player stats via `pybaseball`.
2. **Baseball Game Simulation**: Realistic 9-inning game simulation engine with RE24 run expectancy, base-runner advancement, forced walks, double plays, and sac flies.
3. **Machine Learning Model**: Training an overall 9-position lineup regressor on relative position metrics.
4. **Hybrid ML + Paired Simulation Optimizer**: Screening candidate lineups via ML and refining with paired Monte Carlo seed control.
5. **Statistical Hypothesis Testing**: 10,000-game Monte Carlo simulation with paired t-tests to prove statistical significance and wins added per season.

In [ ]:
# Step 1: Install Dependencies
!pip install -q pybaseball scipy scikit-learn matplotlib pandas numpy

In [ ]:
# Step 2: Core Class Definitions

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from typing import List, Dict, Tuple, Any, Optional
from dataclasses import dataclass
from itertools import permutations
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
import warnings
warnings.filterwarnings('ignore')

# --- CONFIG MANAGER ---
class ConfigManager:
    def __init__(self):
        self.default_simulations = 1000
        self.default_innings = 9
        self.min_plate_appearances = 150
        self.base_matrix = [
            [0.48, 0.25, 0.10],  # Bases empty
            [0.87, 0.48, 0.21],  # Runner on 1st
            [1.12, 0.67, 0.31],  # Runner on 2nd
            [1.38, 0.86, 0.32],  # Runners on 1st & 2nd
            [1.55, 0.96, 0.42],  # Runner on 3rd
            [1.78, 1.31, 0.48],  # Runners on 1st & 3rd
            [2.04, 1.41, 0.67],  # Runners on 2nd & 3rd
            [2.69, 1.61, 0.96]   # Bases loaded
        ]
        self.base_states = [
            (0, 0, 0), (1, 0, 0), (0, 1, 0), (1, 1, 0),
            (0, 0, 1), (1, 0, 1), (0, 1, 1), (1, 1, 1)
        ]

# --- PLAYER MODEL ---
class PlayerModel:
    def __init__(self, player_data: pd.Series, config_manager=None):
        self.name = player_data.get('Name', 'Unknown')
        self.team = player_data.get('Team', player_data.get('Tm', 'Unknown'))
        self.pa = max(int(player_data.get('PA', 1)), 1)
        
        self.singles = float(player_data.get('1B', 0))
        self.doubles = float(player_data.get('2B', 0))
        self.triples = float(player_data.get('3B', 0))
        self.home_runs = float(player_data.get('HR', 0))
        self.walks = float(player_data.get('BB', 0))
        self.hit_by_pitch = float(player_data.get('HBP', 0))
        self.strikeouts = float(player_data.get('SO', 0))
        self.ground_into_double_play = float(player_data.get('GDP', 0))
        
        self.woba = float(player_data.get('wOBA', 0.320))
        self.obp = float(player_data.get('OBP', 0.320))
        self.slg = float(player_data.get('SLG', 0.400))
        self.xwoba = float(player_data.get('xwOBA', self.woba))
        self.xba = float(player_data.get('xBA', 0.250))
        self.xslg = float(player_data.get('xSLG', self.slg))
        self.iso = float(player_data.get('ISO', self.slg - self.obp + 0.05))
        
        self._calculate_derived_stats()
        
    def _calculate_derived_stats(self):
        self.single_rate = self.singles / self.pa
        self.double_rate = self.doubles / self.pa
        self.triple_rate = self.triples / self.pa
        self.hr_rate = self.home_runs / self.pa
        self.walk_rate = self.walks / self.pa
        self.hbp_rate = self.hit_by_pitch / self.pa
        self.strikeout_rate = self.strikeouts / self.pa
        self.gidp_rate = self.ground_into_double_play / self.pa
        self.contact_rate = max(0.0, 1.0 - self.strikeout_rate)
        self.power_score = self.slg - self.obp
        
    def get_outcome_probabilities((self) -> Dict[str, float]:
        sac_fly_prob = min(0.02, max(0.005, self.contact_rate * 0.02))
        groundout_prob = 0.24
        flyout_prob = 0.16
        
        total_positive = (self.single_rate + self.double_rate + self.triple_rate + self.hr_rate +
                         self.walk_rate + self.hbp_rate + self.strikeout_rate + sac_fly_prob +
                         groundout_prob + flyout_prob)
        remaining_out_prob = max(0.0, 1.0 - total_positive)
        
        probs = {
            'single': self.single_rate, 'double': self.double_rate, 'triple': self.triple_rate, 'hr': self.hr_rate,
            'walk': self.walk_rate, 'hbp': self.hbp_rate, 'strikeout': self.strikeout_rate,
            'sac_fly': sac_fly_prob, 'groundout': groundout_prob, 'flyout': flyout_prob, 'out': remaining_out_prob
        }
        total = sum(probs.values())
        return {k: v / total for k, v in probs.items()} if total > 0 else probs
        
    def get_feature_vector(self) -> Dict[str, float]:
        return {
            'wOBA': self.woba, 'OBP': self.obp, 'SLG': self.slg,
            'xwOBA': self.xwoba, 'xBA': self.xba, 'xSLG': self.xslg, 'ISO': self.iso,
            'BB_rate': self.walk_rate, 'K_rate': self.strikeout_rate, 'HR_rate': self.hr_rate,
            'contact_rate': self.contact_rate, 'power_score': self.power_score
        }
        
    def simulate_at_bat(self) -> str:
        probs = self.get_outcome_probabilities()
        return np.random.choice(list(probs.keys()), p=list(probs.values()))

# --- LINEUP MODEL ---
class LineupModel:
    def __init__(self, players: List[PlayerModel], config_manager=None):
        self.players = players[:9]
        self.config_manager = config_manager
        self.lineup_order = [p.name for p in self.players]
        
    def get_player_by_name(self, name: str) -> Optional[PlayerModel]:
        for p in self.players:
            if p.name == name:
                return p
        return None
        
    def set_lineup_order(self, new_order: List[str]):
        self.lineup_order = new_order
        
    def _optimize_traditional(self):
        df_players = list(self.players)
        leadoff = max(df_players, key=lambda p: p.obp)
        two_hole = max([p for p in df_players if p != leadoff], key=lambda p: p.xwoba)
        power = sorted([p for p in df_players if p not in [leadoff, two_hole]], key=lambda p: p.woba, reverse=True)[:2]
        sluggers = sorted([p for p in df_players if p not in [leadoff, two_hole] + power], key=lambda p: p.woba + p.xslg, reverse=True)[:2]
        rest = [p for p in df_players if p not in [leadoff, two_hole] + power + sluggers]
        
        optimized_order = [leadoff.name, two_hole.name] + [p.name for p in power] + [p.name for p in sluggers] + [p.name for p in rest]
        self.set_lineup_order(optimized_order[:9])
        return self
        
    def _optimize_woba(self):
        sorted_p = sorted(self.players, key=lambda p: p.woba, reverse=True)
        optimized_order = [p.name for p in sorted_p]
        self.set_lineup_order(optimized_order[:9])
        return self
        
    def get_lineup_features(self) -> Dict[str, float]:
        features = {}
        for i, name in enumerate(self.lineup_order):
            p = self.get_player_by_name(name)
            if p:
                for k, v in p.get_feature_vector().items():
                    features[f'pos_{i+1}_{k}'] = v
        features['lineup_wOBA_avg'] = np.mean([p.woba for p in self.players])
        features['lineup_OBP_avg'] = np.mean([p.obp for p in self.players])
        features['lineup_SLG_avg'] = np.mean([p.slg for p in self.players])
        features['lineup_power_avg'] = np.mean([p.power_score for p in self.players])
        return features

# --- GAME GENERATOR ---
class GameGenerator:
    def __init__(self, config_manager: ConfigManager):
        self.config = config_manager
        
    def update_game_state(self, bases: List[int], outs: int, outcome: str, player: PlayerModel) -> Tuple[List[int], int, int]:
        runs = 0
        new_bases = bases.copy()
        new_outs = outs
        
        if outcome == 'hr':
            runs += sum(bases) + 1
            new_bases = [0, 0, 0]
        elif outcome == 'single':
            if bases[2]: runs += 1; new_bases[2] = 0
            if bases[1]:
                if outs == 2 or np.random.random() < 0.60:
                    runs += 1; new_bases[1] = 0
                else:
                    new_bases[2] = 1; new_bases[1] = 0
            if bases[0]:
                if outs == 2 and new_bases[2] == 0 and np.random.random() < 0.40:
                    new_bases[2] = 1; new_bases[0] = 0
                else:
                    new_bases[1] = 1; new_bases[0] = 0
            new_bases[0] = 1
        elif outcome == 'double':
            if bases[2]: runs += 1; new_bases[2] = 0
            if bases[1]: runs += 1; new_bases[1] = 0
            if bases[0]:
                if outs == 2 or np.random.random() < 0.40:
                    runs += 1; new_bases[0] = 0
                else:
                    new_bases[2] = 1; new_bases[0] = 0
            new_bases[1] = 1
        elif outcome == 'triple':
            runs += sum(bases)
            new_bases = [0, 0, 1]
        elif outcome in ['walk', 'hbp']:
            if bases[0] and bases[1] and bases[2]:
                runs += 1; new_bases = [1, 1, 1]
            elif bases[0] and bases[1]: new_bases = [1, 1, 1]
            elif bases[0] and bases[2]: new_bases = [1, 1, 1]
            elif bases[0]: new_bases = [1, 1, 0]
            elif bases[1] and bases[2]: new_bases = [1, 1, 1]
            elif bases[1]: new_bases = [1, 1, 0]
            elif bases[2]: new_bases = [1, 0, 1]
            else: new_bases = [1, 0, 0]
        elif outcome == 'sac_fly':
            new_outs += 1
            if outs < 2:
                if bases[2]: runs += 1; new_bases[2] = 0
                if bases[1] and not new_bases[2] and np.random.random() < 0.25:
                    new_bases[2] = 1; new_bases[1] = 0
        elif outcome == 'strikeout':
            new_outs += 1
        elif outcome in ['groundout', 'out']:
            new_outs += 1
            gdp_prob = player.gidp_rate
            if bases[0] and outs < 2 and np.random.random() < gdp_prob:
                new_outs = min(outs + 2, 3)
                new_bases[0] = 0
                if bases[2] and outs == 0: runs += 1; new_bases[2] = 0
                if bases[1]: new_bases[2] = 1; new_bases[1] = 0
            else:
                if bases[2] and outs < 2 and np.random.random() < 0.50: runs += 1; new_bases[2] = 0
                if bases[1] and not new_bases[2] and np.random.random() < 0.35: new_bases[2] = 1; new_bases[1] = 0
                if bases[0] and not new_bases[1]: new_bases[1] = 1; new_bases[0] = 0
        elif outcome == 'flyout':
            new_outs += 1
            if bases[2] and outs < 2 and np.random.random() < 0.70: runs += 1; new_bases[2] = 0
            
        return new_bases, new_outs, runs
        
    def simulate_game(self, lineup: LineupModel, innings: int = 9) -> int:
        total_runs = 0
        batter_idx = 0
        for _ in range(innings):
            outs = 0
            bases = [0, 0, 0]
            while outs < 3:
                p_name = lineup.lineup_order[batter_idx % len(lineup.lineup_order)]
                p = lineup.get_player_by_name(p_name)
                if p is None:
                    outs += 1; batter_idx += 1; continue
                outcome = p.simulate_at_bat()
                bases, outs, runs = self.update_game_state(bases, outs, outcome, p)
                total_runs += runs
                batter_idx += 1
        return total_runs
        
    def evaluate_lineup(self, lineup: LineupModel, n_simulations: int = 1000) -> Dict[str, Any]:
        results = [self.simulate_game(lineup) for _ in range(n_simulations)]
        return {'expected_runs': float(np.mean(results)), 'std_runs': float(np.std(results)), 'simulation_count': n_simulations}

# --- LINEUP ANALYSIS ---
class LineupAnalysis:
    def __init__(self, config_manager: ConfigManager, game_generator: GameGenerator):
        self.config = config_manager
        self.game_generator = game_generator
        self.ml_models = {}
        self.training_data = None
        
    def generate_training_data(self, player_pool: List[PlayerModel], n_lineups: int = 500, n_simulations: int = 1000) -> pd.DataFrame:
        data = []
        print(f"Generating {n_lineups} random lineups ({n_simulations} sims each)...")
        for i in range(n_lineups):
            sub_players = list(np.random.choice(player_pool, size=9, replace=False))
            lineup = LineupModel(sub_players, self.config)
            eval_res = self.game_generator.evaluate_lineup(lineup, n_simulations)
            row = {'lineup_id': i, 'total_runs': eval_res['expected_runs'], **lineup.get_lineup_features()}
            for pos in range(1, 10):
                row[f'pos_{pos}_wOBA_rel'] = row[f'pos_{pos}_wOBA'] - row['lineup_wOBA_avg']
                row[f'pos_{pos}_OBP_rel'] = row[f'pos_{pos}_OBP'] - row['lineup_OBP_avg']
                row[f'pos_{pos}_SLG_rel'] = row[f'pos_{pos}_SLG'] - row['lineup_SLG_avg']
                row[f'pos_{pos}_ISO_rel'] = row[f'pos_{pos}_ISO'] - row['lineup_power_avg']
            data.append(row)
        self.training_data = pd.DataFrame(data)
        return self.training_data
        
    def train_position_models(self, training_data: pd.DataFrame = None):
        if training_data is None: training_data = self.training_data
        all_f = []
        for pos in range(1, 10):
            all_f.extend([f'pos_{pos}_wOBA', f'pos_{pos}_OBP', f'pos_{pos}_SLG', f'pos_{pos}_ISO',
                          f'pos_{pos}_wOBA_rel', f'pos_{pos}_OBP_rel', f'pos_{pos}_SLG_rel', f'pos_{pos}_ISO_rel'])
        cols = [c for c in all_f if c in training_data.columns]
        X = training_data[cols]
        y = training_data['total_runs']
        
        scaler = StandardScaler()
        X_s = scaler.fit_transform(X)
        rf = RandomForestRegressor(n_estimators=100, max_depth=6, random_state=42, n_jobs=-1)
        rf.fit(X_s, y)
        r2 = r2_score(y, rf.predict(X_s))
        
        imp = pd.DataFrame({'feature': cols, 'importance': rf.feature_importances_}).sort_values('importance', ascending=False)
        self.ml_models['overall'] = {'model': rf, 'scaler': scaler, 'features': cols, 'r2': r2, 'importance': imp}
        print(f"ML Model Trained! R2 Score: {r2:.4f}")
        return self.ml_models
        
    def optimize_lineup_ml(self, players: List[PlayerModel]) -> LineupModel:
        if 'overall' not in self.ml_models:
            self.generate_training_data(players, n_lineups=300, n_simulations=500)
            self.train_position_models()
        target_p = players[:9]
        m_info = self.ml_models['overall']
        model, scaler, cols = m_info['model'], m_info['scaler'], m_info['features']
        
        perm_list = list(permutations(target_p))
        X_rows = []
        for perm in perm_list:
            tmp_m = LineupModel(list(perm), self.config)
            fd = tmp_m.get_lineup_features()
            for pos in range(1, 10):
                fd[f'pos_{pos}_wOBA_rel'] = fd[f'pos_{pos}_wOBA'] - fd['lineup_wOBA_avg']
                fd[f'pos_{pos}_OBP_rel'] = fd[f'pos_{pos}_OBP'] - fd['lineup_OBP_avg']
                fd[f'pos_{pos}_SLG_rel'] = fd[f'pos_{pos}_SLG'] - fd['lineup_SLG_avg']
                fd[f'pos_{pos}_ISO_rel'] = fd[f'pos_{pos}_ISO'] - fd['lineup_power_avg']
            X_rows.append([fd.get(c, 0.0) for c in cols])
        
        preds = model.predict(scaler.transform(np.array(X_rows)))
        top_k = min(50, len(preds))
        top_idx = np.argsort(preds)[-top_k:]
        
        # Seed candidate pool with traditional & wOBA orders
        trad_m = LineupModel(target_p, self.config)._optimize_traditional()
        woba_m = LineupModel(target_p, self.config)._optimize_woba()
        base_m = LineupModel(target_p, self.config)
        
        cand_lineups = [list(perm_list[i]) for i in top_idx]
        for sm in [trad_m, woba_m]:
            objs = [base_m.get_player_by_name(n) for n in sm.lineup_order]
            if all(o is not None for o in objs): cand_lineups.append(objs)
            
        ref_lineup = LineupModel([base_m.get_player_by_name(n) for n in trad_m.lineup_order], self.config)
        
        best_delta = -np.inf
        best_cand = list(target_p)
        for cand_p in cand_lineups:
            cm = LineupModel(cand_p, self.config)
            deltas = []
            for g_i in range(1500):
                seed = 5000 + g_i
                np.random.seed(seed); r_c = self.game_generator.simulate_game(cm)
                np.random.seed(seed); r_r = self.game_generator.simulate_game(ref_lineup)
                deltas.append(r_c - r_r)
            m_d = np.mean(deltas)
            if m_d > best_delta:
                best_delta = m_d
                best_cand = cand_p
        
        res_m = LineupModel(best_cand, self.config)
        res_m.set_lineup_order([p.name for p in best_cand])
        return res_m

print('✅ All core classes defined successfully!')

In [ ]:
# Step 3: Fetch Real MLB Team Data or Create Player Pool
import pybaseball as pyb

config = ConfigManager()
game_gen = GameGenerator(config)
analysis = LineupAnalysis(config, game_gen)

print("Fetching 2024 MLB Batting Statistics...")
try:
    raw_df = pyb.batting_stats(2024, qual=150)
    nyy_df = raw_df[raw_df['Team'] == 'NYY'].nlargest(9, 'wOBA')
    yankees_players = [PlayerModel(row, config) for _, row in nyy_df.iterrows()]
    print(f"Successfully fetched {len(yankees_players)} Yankees players from pybaseball.")
except Exception as e:
    print(f"pybaseball fetch error: {e}. Using robust built-in test dataset...")
    names = ["Juan Soto", "Aaron Judge", "Giancarlo Stanton", "Anthony Rizzo", "Gleyber Torres", "Jazz Chisholm Jr.", "DJ LeMahieu", "Alex Verdugo", "Austin Wells"]
    wobas = [0.425, 0.410, 0.350, 0.340, 0.335, 0.330, 0.320, 0.310, 0.300]
    obps  = [0.425, 0.405, 0.335, 0.330, 0.335, 0.320, 0.325, 0.305, 0.295]
    slgs  = [0.590, 0.600, 0.510, 0.440, 0.460, 0.430, 0.390, 0.380, 0.360]
    yankees_players = []
    for i in range(9):
        s = pd.Series({'Name': names[i], 'Team': 'NYY', 'PA': 600, '1B': 90, '2B': 25, '3B': 2, 'HR': 30, 'BB': 75, 'HBP': 5, 'SO': 120, 'GDP': 8, 'wOBA': wobas[i], 'OBP': obps[i], 'SLG': slgs[i], 'xwOBA': wobas[i], 'xBA': obps[i]-0.07, 'xSLG': slgs[i]-0.01, 'ISO': slgs[i]-obps[i]+0.05})
        yankees_players.append(PlayerModel(s, config))

print("Top 5 Yankees Players by wOBA:")
for p in sorted(yankees_players, key=lambda x: x.woba, reverse=True)[:5]:
    print(f"  • {p.name}: wOBA={p.woba:.3f}, OBP={p.obp:.3f}, SLG={p.slg:.3f}")

In [ ]:
# Step 4: ML Regressor Training & Feature Importance

print("Generating multi-player pool for training...")
pool_players = list(yankees_players)
# Add 10 additional varied players to pool
names_extra = ["Shohei Ohtani", "Mookie Betts", "Freddie Freeman", "Yordan Alvarez", "Bryce Harper", "Corey Seager", "Bobby Witt Jr.", "Gunnar Henderson", "Jasson Dominguez", "Ben Rice"]
w_e = [0.405, 0.395, 0.390, 0.385, 0.380, 0.375, 0.370, 0.365, 0.325, 0.315]
o_e = [0.400, 0.390, 0.385, 0.380, 0.375, 0.365, 0.360, 0.355, 0.330, 0.320]
s_e = [0.580, 0.540, 0.520, 0.560, 0.530, 0.510, 0.500, 0.490, 0.420, 0.400]
for i in range(len(names_extra)):
    s = pd.Series({'Name': names_extra[i], 'Team': 'MLB', 'PA': 600, '1B': 85, '2B': 22, '3B': 2, 'HR': 25, 'BB': 70, 'HBP': 4, 'SO': 110, 'GDP': 7, 'wOBA': w_e[i], 'OBP': o_e[i], 'SLG': s_e[i], 'xwOBA': w_e[i], 'xBA': o_e[i]-0.07, 'xSLG': s_e[i]-0.01, 'ISO': s_e[i]-o_e[i]+0.05})
    pool_players.append(PlayerModel(s, config))

training_df = analysis.generate_training_data(pool_players, n_lineups=300, n_simulations=1000)
models_dict = analysis.train_position_models(training_df)
top_features = models_dict['overall']['importance'].head(10)

plt.figure(figsize=(10, 5))
plt.barh(top_features['feature'][::-1], top_features['importance'][::-1], color='navy')
plt.title('ML Lineup Regressor: Top Feature Importances')
plt.xlabel('Feature Importance Score')
plt.grid(True, linestyle='--', alpha=0.6)
plt.show()

In [ ]:
# Step 5: Generate Lineup Orders & Run Hybrid ML+Simulation Optimizer

print("Generating Lineups for Comparison...")

# 1. ML Optimized Lineup
ml_lineup_model = analysis.optimize_lineup_ml(yankees_players)
ml_order = ml_lineup_model.lineup_order

# 2. Traditional Lineup
trad_lineup_model = LineupModel(yankees_players, config)._optimize_traditional()
trad_order = trad_lineup_model.lineup_order

# 3. wOBA-Sorted Lineup
woba_lineup_model = LineupModel(yankees_players, config)._optimize_woba()
woba_order = woba_lineup_model.lineup_order

# 4. Control Lineup (Suboptimal Reverse wOBA)
sorted_y = sorted(yankees_players, key=lambda p: p.woba, reverse=True)
ctrl_order = [p.name for p in reversed(sorted_y)]
ctrl_lineup_model = LineupModel(yankees_players, config)
ctrl_lineup_model.set_lineup_order(ctrl_order)

print("\n--- LINEUP COMPARISON ORDERINGS ---")
print("ML-Optimized Lineup: ", ml_order)
print("Traditional Lineup:  ", trad_order)
print("wOBA-Sorted Lineup:  ", woba_order)
print("Control Lineup:      ", ctrl_order)

In [ ]:
# Step 6: 10,000-Game Monte Carlo Simulation & Paired t-Test Validation

N_GAMES = 10000
print(f"Running Monte Carlo Simulation across {N_GAMES:,} games per lineup...")

ml_runs, trad_runs, woba_runs, ctrl_runs = [], [], [], []

for i in range(N_GAMES):
    seed = 10000 + i
    
    np.random.seed(seed); ml_runs.append(game_gen.simulate_game(ml_lineup_model))
    np.random.seed(seed); trad_runs.append(game_gen.simulate_game(trad_lineup_model))
    np.random.seed(seed); woba_runs.append(game_gen.simulate_game(woba_lineup_model))
    np.random.seed(seed); ctrl_runs.append(game_gen.simulate_game(ctrl_lineup_model))

mean_ml = np.mean(ml_runs)
mean_trad = np.mean(trad_runs)
mean_woba = np.mean(woba_runs)
mean_ctrl = np.mean(ctrl_runs)

_, p_val_ctrl = stats.ttest_rel(ml_runs, ctrl_runs, alternative='greater')
_, p_val_trad = stats.ttest_rel(ml_runs, trad_runs, alternative='greater')

wins_ctrl = (mean_ml - mean_ctrl) * 162 / 10.0
wins_trad = (mean_ml - mean_trad) * 162 / 10.0

print("\n=======================================================")
print("          MONTE CARLO SIMULATION RESULTS               ")
print("=======================================================")
print(f"ML-Optimized Lineup Expected Runs: {mean_ml:.4f}")
print(f"Traditional Lineup Expected Runs:  {mean_trad:.4f}")
print(f"wOBA-Sorted Lineup Expected Runs:  {mean_woba:.4f}")
print(f"Control (Suboptimal) Expected Runs: {mean_ctrl:.4f}")
print("-------------------------------------------------------")
print(f"Paired t-test p-value (ML vs Control):     {p_val_ctrl:.5f}")
print(f"Paired t-test p-value (ML vs Traditional): {p_val_trad if not np.isnan(p_val_trad) else 0.5:.5f}")
print(f"Wins Added Over Control per Season:        {wins_ctrl:+.2f} wins")
print(f"Wins Added Over Traditional per Season:    {wins_trad:+.2f} wins")
print("=======================================================")

# Bar plot visualization
methods = ['Control', 'ML-Optimized', 'Traditional', 'wOBA-Sorted']
means = [mean_ctrl, mean_ml, mean_trad, mean_woba]
colors = ['#d9534f', '#5cb85c', '#0275d8', '#f0ad4e']

plt.figure(figsize=(9, 5))
bars = plt.bar(methods, means, color=colors, width=0.55)
plt.ylim(min(means) - 0.2, max(means) + 0.2)
plt.ylabel('Expected Runs per Game')
plt.title('Lineup Optimization Expected Runs (10,000 Games)')
for bar in bars:
    yval = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2, yval + 0.02, f"{yval:.4f}", ha='center', va='bottom', fontweight='bold')
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()